# Water-ski CV demo on a Colab GPU

Runs the slow parts of the demo on a GPU:
1. **Re-label** buoys with Grounding DINO on many more frames (every 0.2 s instead of 0.5 s), which takes about 10 min on a T4.
2. **Train** the fast buoy detector (YOLO11s, 1280 px). This version is bigger and more accurate than the CPU one, and takes about 15–25 min.
3. **Benchmark** every model on the GPU to get the time and cost per tool.
4. **SAM 2** tracking (detector-guided and memory modes) at GPU speed.
5. **Full annotated demo** on all test videos.
6. **Save** a results zip to your Google Drive, then share the link back in the chat.

**Before you start:** *Runtime → Change runtime type → T4 GPU* (or better), then *Runtime → Run all*.
If the GitHub repo is private, first add a Colab secret named `GH_TOKEN` (key icon on the left) holding a GitHub token with read access.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import os
try:
    from google.colab import userdata
    tok = userdata.get('GH_TOKEN')
except Exception:
    tok = None
url = f"https://{tok}@github.com/lytvynenkomaksim/CV_test.git" if tok else "https://github.com/lytvynenkomaksim/CV_test.git"
if not os.path.exists('/content/CV_test'):
    !git clone --depth 1 -b main {url} /content/CV_test
%cd /content/CV_test
!git log --oneline | head -3

In [ ]:
%%capture install_log
!apt-get -qq install -y libegl1 libgles2
!pip install -q ultralytics transformers accelerate rtmlib onnxruntime-gpu lap mediapipe "git+https://github.com/ultralytics/CLIP.git"


In [ ]:
!bash tools/download_models.sh
!yolo settings weights_dir=/content/CV_test/models > /dev/null
os.environ['HF_HOME'] = '/content/CV_test/models/hf'
import torch; print(torch.__version__, torch.cuda.get_device_name(0))

## 1. Re-label buoys with Grounding DINO (GPU)
Same holdout videos as on CPU (2 Italy clips + the low-quality Greece clip). The YouTube video stays unseen.

In [ ]:
!python tools/autolabel_gdino.py --out data/buoy_dataset_gpu --every 0.2 \
    --videos data/videos/front_view/*.mp4 data/videos/drive_extra/*.mp4 \
    --holdout "Vincenzo Marino 6 at 35" "Lucas Cornale 6 at 35" "Aaron_Davis" 2>&1 | tail -3
!for s in train val; do echo $s $(ls data/buoy_dataset_gpu/images/$s | wc -l) images, $(cat data/buoy_dataset_gpu/labels/$s/*.txt | wc -l) buoy boxes; done

## 2. Train the fast buoy detector

In [ ]:
!python tools/train_buoy_yolo.py --data data/buoy_dataset_gpu/data.yaml --model yolo11s.pt \
    --epochs 80 --imgsz 1280 --batch 16 --workers 4 2>&1 | grep -E "all|saved|Epoch" | tail -5

In [ ]:
import pandas as pd
r = pd.read_csv('runs/train/buoy/results.csv'); r.columns = [c.strip() for c in r.columns]
r[['epoch','metrics/precision(B)','metrics/recall(B)','metrics/mAP50(B)','metrics/mAP50-95(B)']].tail(5)

## 3. Benchmark every model on the GPU (low / medium / high quality + YouTube)

In [ ]:
!python tools/bench.py \
  data/videos/drive_extra/GreecePro_MenRd3__Aaron_Davis_Rd3_6at12.mp4 \
  "data/videos/front_view/Vincenzo Marino 6 at 32 off.mp4" \
  data/videos/drive_extra/ItalyPro_OpenWomenRd2__Julie_Mishler_6at32_Rd2.mp4 \
  data/videos/youtube/youtube_regina_jaquess.mp4 \
  --frames 30 --out data/benchmarks/bench_gpu.json 2>&1 | grep -v -i warn | tail -40

## 4. SAM 2 at GPU speed (detector-guided vs memory, tiny/small/base)

In [ ]:
V = "data/videos/front_view/Vincenzo Marino 6 at 32 off.mp4"
for m in ['sam2.1_t.pt', 'sam2.1_s.pt', 'sam2.1_b.pt']:
    !python tools/sam2_track.py "{V}" --start 9 --dur 5 --model {m} --mode guided --out runs/sam2_gpu 2>&1 | tail -1
!python tools/sam2_track.py "{V}" --start 9 --dur 5 --model sam2.1_t.pt --mode memory --out runs/sam2_gpu 2>&1 | tail -1

## 5. Full annotated demo on all test videos

In [ ]:
!python tools/run_all.py --buoy yolo-buoy --pose yolo-s --out runs/demo_gpu 2>&1 | grep -v -i warn | tail -25

## 6. Save results to Google Drive
Afterwards, open `MyDrive/cv_test_results/`, share the zip (*Anyone with the link → Viewer*) and paste the link in the chat.

In [ ]:
!mkdir -p results && cp models/buoy_yolo11s.pt models/buoy_yolo11s.onnx results/ 2>/dev/null; \
 cp -r runs/train/buoy results/train_buoy && rm -rf results/train_buoy/weights; \
 cp -r data/benchmarks runs/sam2_gpu runs/demo_gpu results/; \
 cp -r data/buoy_dataset_gpu results/ ; rm -f results/buoy_dataset_gpu/data_resolved.yaml
!cd results && zip -qr ../cv_test_results.zip . && cd .. && ls -lh cv_test_results.zip
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/cv_test_results && cp cv_test_results.zip /content/drive/MyDrive/cv_test_results/
print('saved to MyDrive/cv_test_results/cv_test_results.zip')